In [1]:
import librosa as lb
import numpy as np
import pickle
from eval_tools import getGroundTruthTimestamps
import utils.constants as constants
import plotly.graph_objs as go

In [2]:
s = 1

dtw_hyp_file = f"experiments/DTW/s{s}/hyp.npy"
noa_hyp_file = f"experiments/NOA/s{s}/hyp.npy"
noa_tsm_file = f"experiments/NOA/s{s}/tsm.npy"

# load
dtw_hyp = np.load(dtw_hyp_file)
noa_hyp = np.load(noa_hyp_file)
noa_tsm = np.load(noa_tsm_file)

# load the ground truth timestamps
query_annot_file = f'scenarios/s{s}/query.beats'
ref_annot_file = f'scenarios/s{s}/ref.beats'
gt = getGroundTruthTimestamps(query_annot_file, ref_annot_file).T

In [3]:
pair_txt_file = f'scenarios/s{s}/pair.txt'
pair_txt = open(pair_txt_file, 'r').readlines()
pair_txt = [line.strip().split() for line in pair_txt]
id1 = pair_txt[0][0]
id2 = pair_txt[0][1]

# load the chroma features
audio_path_1 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id1}.wav'
audio_path_2 = f'Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/{id2}.wav'

# load the features
f1 = np.load(f'features/chroma_stft/{id1}.npy')
f2 = np.load(f'features/chroma_stft/{id2}.npy')

In [4]:
from noa_kalman import alignNOAKalman
Q = np.array([[1000, 0], [0, 0.0001]])
R = np.array([[0.01]])
results = alignNOAKalman(f1, f2, Q = Q, R = R)

In [5]:
# plot results.x over time
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.arange(0, len(results.velocity_history)), y=results.velocity_history, mode='lines', name='NOA Kalman'
))
fig.show()

In [6]:
fig = go.Figure()
fig.update_layout(
    title='Alignment and TSM Paths',
    xaxis_title='Query Time (s)',
    yaxis_title='Reference Time (s)',
    width=900,
    height=700
)
fig.add_trace(go.Scatter(
    x=noa_hyp[0], y=noa_hyp[1], mode='lines', name='NOA Hyp'
))
fig.add_trace(go.Scatter(
    x=results.get_path()[0, :], y=results.get_path()[1, :], mode='lines', name='NOA Kalman'
))
fig.add_trace(go.Scatter(
    x=gt[0], y=gt[1], mode='markers', name='Ground Truth'
))
fig.show()